# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaifLatki/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

The strongest signal in this dataset is that traffic and page-quality metrics are extremely heavy-tailed. A few pages are huge, while most are modest; that matters because a raw average can be dominated by a handful of giant pages and hide the real operating pattern.

I therefore looked at medians and tail summaries before drawing conclusions, and I kept the verdicts tied to sample size so small buckets do not get dressed up as meaningful signals.


In [ ]:
import pandas as pd
import numpy as np

# Read the prepared feature vector and keep the key columns used in the signal audit.
df = pd.read_csv('data/processed/refresh_feature_vector.csv')

# Heavy-tail checks on the biggest traffic and quality columns.
summary = df[['impressions_90d', 'sessions_90d', 'word_count', 'ctr', 'avg_position', 'days_since_last_update']].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
print(summary[['count', 'mean', '50%', 'std', '95%', '99%', 'max']].to_string())

print('\nPosition tier median CTR:')
print(df.groupby('position_tier')['ctr'].median().sort_index().to_string())
print('\nContent type median impressions:')
print(df.groupby('content_type')['impressions_90d'].median().sort_values(ascending=False).to_string())
print('\nWord count tier median impressions:')
print(df.groupby('word_count_tier')['impressions_90d'].median().sort_index().to_string())


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Signal test #1: Longer pages carry more traffic
# Claim: pages with more words tend to have higher impressions.
# Test: compare median impressions by word_count_tier, then check the direction of the pattern.
word_count_bucket = df.groupby('word_count_tier', observed=False).agg(
    rows=('content_id', 'count'),
    median_impressions=('impressions_90d', 'median'),
    median_ctr=('ctr', 'median'),
).sort_index()
print(word_count_bucket.to_string())
print('\nVerdict: MIXED')
print('Interpretation: very short pages are weak, but the strongest traffic is not a simple linear function of length; mid-to-long pages do better, while extremely short pages remain weak and noisy.')

# Signal test #2: Better position is associated with stronger CTR.
# Claim: pages ranking near the top convert more efficiently than deeper pages.
# Test: compare median CTR across position_tier buckets.
position_bucket = df.groupby('position_tier', observed=False).agg(
    rows=('content_id', 'count'),
    median_ctr=('ctr', 'median'),
    median_impressions=('impressions_90d', 'median'),
).sort_index()
print('\nPosition tier test:')
print(position_bucket.to_string())
print('\nVerdict: CONFIRMED')
print('Interpretation: pages on page one have materially higher CTR than deep pages, but the top-3 bucket is sparse and should be treated as a small-sample case.')

# Signal test #3: Freshness is not a strong standalone driver of traffic in the visible pages slice.
# Claim: stale pages should be weaker than fresh pages.
# Test: compare visible pages (>=500 impressions) across freshness buckets.
visible = df[df['impressions_90d'] >= 500].copy()
visible['freshness_bucket'] = pd.cut(
    visible['days_since_last_update'],
    bins=[0, 30, 90, 180, 365, 999],
    labels=['0-30', '31-90', '91-180', '181-365', '365+'],
    right=False,
)
print('\nFreshness bucket comparison for visible pages:')
print(visible.groupby('freshness_bucket', observed=False).agg(
    rows=('content_id', 'count'),
    median_ctr=('ctr', 'median'),
    median_impressions=('impressions_90d', 'median'),
).to_string())
print('\nVerdict: FALSE')
print('Interpretation: among large, visible pages, age does not separate the winners cleanly; the largest signal is visibility itself, not age alone.')


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Flag-linked test: does a visible page on page one with low CTR look like a likely refresh opportunity?
# This is the same logic that the baseline rule uses to surface content that is visible but underperforming.
visible_page_one = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
low_ctr_flag = visible_page_one['ctr'] < 0.5
print('Visible pages on page one or near page one:', len(visible_page_one))
print('Low-CTR share of visible page-one pages:', low_ctr_flag.mean())
print('Median CTR of visible page-one pages:', visible_page_one['ctr'].median())
print('Median engagement rate of visible page-one low-CTR pages:', visible_page_one.loc[low_ctr_flag, 'engagement_rate'].median())
print('Median scroll rate of visible page-one low-CTR pages:', visible_page_one.loc[low_ctr_flag, 'scroll_rate'].median())
print('Median impressions of visible page-one low-CTR pages:', visible_page_one.loc[low_ctr_flag, 'impressions_90d'].median())
print('\nInterpretation: the data does support the assumption that visible pages with low CTR are a meaningful refresh candidate set, but low CTR alone is not sufficient; the strongest action signals come from high visibility plus weak engagement and page-one positioning.')


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# The practical takeaway for a content team.
print('A useful human summary:')
print('1) Traffic is not driven by page length alone; the strongest pages are visible and mid-length to long, but the signal is mixed and not monotonic.')
print('2) Page-one performance is a real operational signal: pages higher in rankings have meaningfully better CTR than deep pages, which is consistent with the idea that ranking quality matters.')
print('3) For refresh work, the highest-value candidates are visible pages near page one that still show weak engagement or low CTR; that pattern is more actionable than age alone.')


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
